## 🚀 Tesla Stock News Alert Automation

This project automatically checks the **Tesla (TSLA)** stock price every day.  
If the price changes by more than **5%**, it fetches the latest related news and sends them to your phone via **SMS** using **Twilio**.

---

## 🧩 Features

- 📊 Fetches stock data using the **Alpha Vantage API**
- 🗞️ Retrieves news articles from the **News API**
- 📱 Sends SMS alerts with **Twilio**
- ⚙️ Calculates price changes and filters significant movement (±5%)

---

## ⚙️ Setup Instructions

### 1️⃣ Install Required Libraries
```bash
pip install requests twilio


In [1]:
import requests
from twilio.rest import Client

STOCK_NAME = "TSLA"
COMPANY_NAME = "Tesla Inc"

STOCK_API_KEY = "AIVA6I4KLKNUXPUD"
NEWS_API_KEY = "aeedd4bb57a94e52ba24954dd19dcb02"
STOCK_ENDPOINT = "https://www.alphavantage.co/query"
NEWS_ENDPOINT = "https://newsapi.org/v2/everything"

TWILIO_SID = "YOUR_TWILIO_SID"
TWILIO_AUTH_TOKEN = "YOUR_TWILIO_AUTH_TOKEN"
TWILIO_FROM_NUMBER = "YOUR_TWILIO_PHONE_NUMBER"
TWILIO_TO_NUMBER = "YOUR_PERSONAL_PHONE_NUMBER"

stock_params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": STOCK_NAME,
    "apikey": STOCK_API_KEY,
}

response = requests.get(STOCK_ENDPOINT, params=stock_params)
response.raise_for_status()
data = response.json()["Time Series (Daily)"]
data_list = [value for (key, value) in data.items()]

yesterday_data = data_list[0]
yesterday_closing_price = float(yesterday_data["4. close"])

day_before_yesterday_data = data_list[1]
day_before_yesterday_closing_price = float(day_before_yesterday_data["4. close"])

difference = abs(yesterday_closing_price - day_before_yesterday_closing_price)
percent_diff = (difference / day_before_yesterday_closing_price) * 100

up_down = "🔺" if yesterday_closing_price > day_before_yesterday_closing_price else "🔻"

if percent_diff > 5:
    news_params = {
        "qInTitle": COMPANY_NAME,
        "apiKey": NEWS_API_KEY,
        "language": "en",
        "sortBy": "publishedAt"
    }

    news_response = requests.get(NEWS_ENDPOINT, params=news_params)
    news_response.raise_for_status()
    articles = news_response.json()["articles"]

    top_3_articles = articles[:3]
    formatted_articles = [
        f"{STOCK_NAME}: {up_down}{round(percent_diff)}%\nHeadline: {article['title']}\nBrief: {article['description']}"
        for article in top_3_articles
    ]

    client = Client(TWILIO_SID, TWILIO_AUTH_TOKEN)
    for message_body in formatted_articles:
        message = client.messages.create(
            body=message_body,
            from_=TWILIO_FROM_NUMBER,
            to=TWILIO_TO_NUMBER
        )
        print(message.status)


ModuleNotFoundError: No module named 'twilio'

## 🧠 How It Works
Fetch Stock Prices — Retrieves daily Tesla stock data from Alpha Vantage.

Calculate Change — Computes the percent difference between the last two closing prices.

Fetch News — Gets the top 3 recent Tesla-related articles if change > 5%.

Send Alerts — Sends the article headlines and summaries via SMS.